In [1]:
!sudo apt-get install cmake build-essential pkg-config libgoogle-perftools-dev -y

Reading package lists... Done
Building dependency tree       
Reading state information... Done
pkg-config is already the newest version (0.29.1-0ubuntu4).
build-essential is already the newest version (12.8ubuntu1.1).
cmake is already the newest version (3.16.3-1ubuntu1.20.04.1).
The following additional packages will be installed:
  libgoogle-perftools4 liblzma-dev libtcmalloc-minimal4 libunwind-dev
Suggested packages:
  liblzma-doc
The following NEW packages will be installed:
  libgoogle-perftools-dev libgoogle-perftools4 liblzma-dev
  libtcmalloc-minimal4 libunwind-dev
0 upgraded, 5 newly installed, 0 to remove and 40 not upgraded.
Need to get 1352 kB of archives.
After this operation, 10.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu focal/main amd64 libtcmalloc-minimal4 amd64 2.7-1ubuntu2 [93.0 kB]
Get:2 http://archive.ubuntu.com/ubuntu focal/main amd64 libgoogle-perftools4 amd64 2.7-1ubuntu2 [195 kB]
Get:3 http://archive.ubuntu.com/ubuntu foc

In [2]:
import os

In [3]:
! git clone https://github.com/google/sentencepiece.git 
os.chdir('sentencepiece')
! mkdir build
os.chdir('build')
! cmake ..
! make -j $(nproc)
! sudo make install
! sudo ldconfig -v

Cloning into 'sentencepiece'...
remote: Enumerating objects: 4823, done.
remote: Counting objects: 100% (1857/1857), done.
remote: Compressing objects: 100% (197/197), done.
remote: Total 4823 (delta 1685), reused 1660 (delta 1660), pack-reused 2966
Receiving objects: 100% (4823/4823), 26.68 MiB | 17.13 MiB/s, done.
Resolving deltas: 100% (3338/3338), done.
cmake: /opt/conda/lib/libcurl.so.4: no version information available (required by cmake)
-- VERSION: 0.2.00
-- The C compiler identification is GNU 9.4.0
-- The CXX compiler identification is GNU 9.4.0
-- Check for working C compiler: /usr/bin/cc
-- Check for working C compiler: /usr/bin/cc -- works
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Detecting C compile features
-- Detecting C compile features - done
-- Check for working CXX compiler: /usr/bin/c++
-- Check for working CXX compiler: /usr/bin/c++ -- works
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Detecting 

In [4]:
os.chdir('/kaggle/working')
!ls

sentencepiece


In [5]:
import re

In [6]:
import spacy
import re

def english_preprocess(s: str):
    nlp = spacy.load("en_core_web_sm")
    
    # Replace semicolons and colons with commas
    s = re.sub(r'[;:]+', ",", s)
    # Remove angle brackets
    s = re.sub(r'[<>]+', "", s)
    
    # Replace parentheses with "<emo>"
    s = re.sub(r'\([\sa-zA-Z]+\)', "<GS> ", s)
    # Remove unwanted characters
    s = re.sub(r"[^\sa-zA-Z0-9?.!,'<>-]+", "", s)
    # Replace multiple spaces with a single space
    s = re.sub(r'[" "]+', " ", s)

    doc = nlp(s.lower())
    corrected_text = ''
    capitalize_next = False
    for sent in doc.sents:
        sentence_text = ''
        for i, token in enumerate(sent):
            # Capitalize the first letter of the first word
            if i == 0:
                sentence_text += token.text.capitalize()
            elif token.text in [',', '?', '.', '!']:
                capitalize_next = True
                sentence_text += token.text
            elif capitalize_next:
                # Capitalize the first letter of the next word after a comma
                sentence_text += ' ' + token.text.capitalize()
                capitalize_next = False
            elif token.pos_ in ('PROPN', 'NNP', 'NNPS'):
                sentence_text += ' ' + token.text.capitalize()
            else:
                sentence_text += ' ' + token.text.lower()
        corrected_text += ' ' + sentence_text   
    corrected_text = re.sub(r"(\s')+", "'", corrected_text)
    corrected_text = re.sub(r"(\s[nN]'t)+", "n't", corrected_text)
    corrected_text = re.sub(r"([\s]+-[\s]+)+", "-", corrected_text)
    corrected_text = re.sub(r"(<[\s]+[a-zA-Z]+[\s]+>)+", " <GS> ", corrected_text)
    corrected_text = re.sub(r'[" "]+', " ", corrected_text)
    return corrected_text.strip()

# Example usage
input_text = """we got off the exit, we found a shoney's restaurant.
low-cost family restaurant chain, for those of you who don't know it.
we went in and sat down at the booth, and the waitress came over, made a big commotion over tipper. (laughter) she took our order, and then went to the couple in the booth next to us, and she lowered her voice so much, i had to really strain to hear what she was saying.
and she said "yes, that's former vice president al gore and his wife, tipper." and the man said, "he's come down a long way, hasn't he?" (laughter)
there's been kind of a series of epiphanies.
the very next day, continuing the totally true story, i got on a g-v to fly to africa to make a speech in nigeria, in the city of lagos, on the topic of energy.
and i began the speech by telling them the story of what 9.8 had just happened the day before in nashville.
and i told it pretty much the same way i've just shared  is.  A it with you: tipper and i were driving ours"""

output_text = english_preprocess(input_text)
print(output_text)


We got off the exit, We found a shoney's restaurant. 
 Low-cost family restaurant chain, For those of you who don't know it. 
 We went in and sat down at the booth, And the waitress came over, Made a big commotion over tipper. <GS> she took our order, And then went to the couple in the booth next to us, And she lowered her voice so much, I had to really strain to hear what she was saying. 
 And she said yes, That's former vice President Al Gore and his wife, Tipper. And The man said, He's come down a long way, Hasn't he? <GS> 
 there's been kind of a series of epiphanies. 
 The very next day, Continuing the totally true story, I got on a g-v to fly to Africa to make a speech in Nigeria, In the city of lagos, On the topic of energy. 
 And i began the speech by telling them the story of what 9.8 had just happened the day before in Nashville. 
 And i told it pretty much the same way i've just shared is. A It with you, Tipper and i were driving ours


In [7]:
from tqdm import tqdm

In [8]:
with open('/kaggle/input/must-c-en-ar/train/txt/train_en.txt', 'rt') as f:
    text = ''
    lines = f.readlines()
    print(len(lines))
    batch_size = 600
    for i in tqdm(range(0, len(lines), batch_size)):
        batch = '\n'.join(lines[i: i+batch_size])
        batch = english_preprocess(batch)
        text +=  '\n' + batch

212085


100%|██████████| 354/354 [20:21<00:00,  3.45s/it]


In [9]:
print(len(text.split('\n')))
type(text)

423815


str

In [10]:
"<GS> "*10

'<GS> <GS> <GS> <GS> <GS> <GS> <GS> <GS> <GS> <GS> '

In [11]:
with open('bn-speech.txt', 'w') as f:
    f.write(text + f'\n {" ".join(map(str, list(range(0, 100))))}')

## Usage: spm_train [options] files

   **--help** (show help)  type: bool default: false
   
   **--version** (show version)  type: bool default: false
   
   **--minloglevel** (Messages logged at a lower level than this don't actually get logged anywhere)  type: int default: 0
   
   **--input** (comma separated list of input sentences)  type: std::string default: ""
   
   **--input_format** (Input format. Supported format is `text` or `tsv`.)  type: std::string default: ""
   
   **--model_prefix** (output model prefix)  type: std::string default: ""
   
   **--model_type** (model algorithm: unigram, bpe, word or char)  type: std::string default: "unigram"
   
   **--vocab_size** (vocabulary size)  type: int32 default: 8000
   
   **--accept_language** (comma-separated list of languages this model can accept)  type: std::string default: ""
   
   **--self_test_sample_size** (the size of self test samples)  type: int32 default: 0
   
   **--character_coverage** (character coverage to determine the minimum symbols)  type: double default: 0.9995
   
   **--input_sentence_size** (maximum size of sentences the trainer loads)  type: std::uint64_t default: 0
   
   **--shuffle_input_sentence** (Randomly sample input sentences in advance. Valid when --input_sentence_size > 0)  type: bool default: true
   
   **--seed_sentencepiece_size** (the size of seed sentencepieces)  type: int32 default: 1000000
   
   **--shrinking_factor** (Keeps top shrinking_factor pieces with respect to the loss)  type: double default: 0.75
   
   **--num_threads** (number of threads for training)  type: int32 default: 16
   
   **--num_sub_iterations** (number of EM sub-iterations)  type: int32 default: 2
   
   **--max_sentencepiece_length** (maximum length of sentence piece)  type: int32 default: 16
   
   **--max_sentence_length** (maximum length of sentence in byte)  type: int32 default: 4192
   
   **--split_by_unicode_script** (use Unicode script to split sentence pieces)  type: bool default: true
   
   **--split_by_number** (split tokens by numbers (0-9))  type: bool default: true
   
   **--split_by_whitespace** (use a white space to split sentence pieces)  type: bool default: true
   
   **--split_digits** (split all digits (0-9) into separate pieces)  type: bool default: false
   
   **--pretokenization_delimiter** (specifies the delimiter of pre-tokenization)  type: std::string default: ""
   
   **--treat_whitespace_as_suffix** (treat whitespace marker as suffix instead of prefix.)  type: bool default: false
   
   **--allow_whitespace_only_pieces** (allow pieces that only contain (consecutive) whitespace tokens)  type: bool default: false
   
   **--control_symbols** (comma separated list of control symbols)  type: std::string default: ""
   
   **--control_symbols_file** (load control_symbols from file.)  type: std::string default: ""
   
   **--user_defined_symbols** (comma separated list of user defined symbols)  type: std::string default: ""
   
   **--user_defined_symbols_file** (load user_defined_symbols from file.)  type: std::string default: ""
   
   **--required_chars** (UTF8 characters in this flag are always used in the character set regardless of --character_coverage)  type: std::string default: ""
   
   **--required_chars_file** (load required_chars from file.)  type: std::string default: ""
   
   **--byte_fallback** (decompose unknown pieces into UTF-8 byte pieces)  type: bool default: false
   
   **--vocabulary_output_piece_score** (Define score in vocab file)  type: bool default: true
   
   **--normalization_rule_name** (Normalization rule name. Choose from nfkc or identity)  type: std::string default: "nmt_nfkc"
   
   **--normalization_rule_tsv** (Normalization rule TSV file. )  type: std::string default: ""
   
   **--denormalization_rule_tsv** (Denormalization rule TSV file.)  type: std::string default: ""
   
   **--add_dummy_prefix** (Add dummy whitespace at the beginning of text)  type: bool default: true
   
   **--remove_extra_whitespaces** (Removes leading, trailing, and duplicate internal whitespace)  type: bool default: true
   
   **--hard_vocab_limit** (If set to false, --vocab_size is considered as a soft limit.)  type: bool default: true
   
   **--use_all_vocab** (If set to true, use all tokens as vocab. Valid for word/char models.)  type: bool default: false
   
   **--unk_id** (Override UNK (\<unk>) id.)  type: int32 default: 0
    
   **--bos_id** (Override BOS (\<s>) id. Set -1 to disable BOS.)  type: int32 default: 1
    
   **--eos_id** (Override EOS (\</s>) id. Set -1 to disable EOS.)  type: int32 default: 2
    
   **--pad_id** (Override PAD (\<pad>) id. Set -1 to disable PAD.)  type: int32 default: -1
    
   **--unk_piece** (Override UNK (\<unk>) piece.)  type: std::string default: "\<unk>"
    
   **--bos_piece** (Override BOS (\<s>) piece.)  type: std::string default: "\<s>"
    
   **--eos_piece** (Override EOS (\</s>) piece.)  type: std::string default: "\</s>"
    
   **--pad_piece** (Override PAD (\<pad>) piece.)  type: std::string default: "\<pad>"
    
   **--unk_surface** (Dummy surface string for \<unk>. In decoding \<unk> is decoded to `unk_surface`.)  type: std::string default: " ⁇ "
    
   **--train_extremely_large_corpus** (Increase bit depth for unigram tokenization.)  type: bool default: false
    
   **--random_seed** (Seed value for random generator.)  type: uint32 default: 4294967295
    
   **--enable_differential_privacy** (Whether to add DP while training. Currently supported only by UNIGRAM model.)  type: bool default: false
    
   **--differential_privacy_noise_level** (Amount of noise to add for DP)  type: float default: 0
    
   **--differential_privacy_clipping_threshold** (Threshold for clipping the counts for DP)  type: std::uint64_t default: 0


In [12]:
!spm_train --input=bn-speech.txt --input_format=text --model_prefix=spm_100  --vocab_size=100 --model_type=bpe \
--character_coverage=1 --random_seed=10  num_threads=4 --pad_id=0 --unk_id=3 --bos_id=1 --eos_id=2 \
--allow_whitespace_only_pieces=True --train_extremely_large_corpus=True --hard_vocab_limit=False \
--max_sentencepiece_length=5 --num_sub_iterations=5 --split_by_number=False --max_sentence_length=30000

sentencepiece_trainer.cc(77) LOG(INFO) Starts training with : 
trainer_spec {
  input: bn-speech.txt
  input_format: text
  model_prefix: spm_100
  model_type: BPE
  vocab_size: 100
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 30000
  num_threads: 16
  num_sub_iterations: 5
  max_sentencepiece_length: 5
  split_by_unicode_script: 1
  split_by_number: 0
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 1
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 1
  hard_vocab_limit: 0
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_privacy_noise_level: 0
  dif

In [13]:
!spm_train --input=bn-speech.txt --input_format=text --model_prefix=spm_300  --vocab_size=300 --model_type=bpe \
--character_coverage=1 --random_seed=10  num_threads=4 --pad_id=0 --unk_id=3 --bos_id=1 --eos_id=2 \
--allow_whitespace_only_pieces=True --train_extremely_large_corpus=True --hard_vocab_limit=False \
--max_sentencepiece_length=5 --num_sub_iterations=5 --split_by_number=False --max_sentence_length=30000

sentencepiece_trainer.cc(77) LOG(INFO) Starts training with : 
trainer_spec {
  input: bn-speech.txt
  input_format: text
  model_prefix: spm_300
  model_type: BPE
  vocab_size: 300
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 30000
  num_threads: 16
  num_sub_iterations: 5
  max_sentencepiece_length: 5
  split_by_unicode_script: 1
  split_by_number: 0
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 1
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 1
  hard_vocab_limit: 0
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_privacy_noise_level: 0
  dif

In [14]:
!spm_train --input=bn-speech.txt --input_format=text --model_prefix=spm_500  --vocab_size=500 --model_type=bpe \
--character_coverage=1 --random_seed=10  num_threads=4 --pad_id=0 --unk_id=3 --bos_id=1 --eos_id=2 \
--allow_whitespace_only_pieces=True --train_extremely_large_corpus=True --hard_vocab_limit=False \
--max_sentencepiece_length=5 --num_sub_iterations=5 --split_by_number=False --max_sentence_length=30000

sentencepiece_trainer.cc(77) LOG(INFO) Starts training with : 
trainer_spec {
  input: bn-speech.txt
  input_format: text
  model_prefix: spm_500
  model_type: BPE
  vocab_size: 500
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 30000
  num_threads: 16
  num_sub_iterations: 5
  max_sentencepiece_length: 5
  split_by_unicode_script: 1
  split_by_number: 0
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 1
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 1
  hard_vocab_limit: 0
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_privacy_noise_level: 0
  dif

In [15]:
!spm_train --input=bn-speech.txt --input_format=text --model_prefix=spm_800  --vocab_size=800 --model_type=bpe \
--character_coverage=1 --random_seed=10  num_threads=4 --pad_id=0 --unk_id=3 --bos_id=1 --eos_id=2 \
--allow_whitespace_only_pieces=True --train_extremely_large_corpus=True --hard_vocab_limit=False \
--max_sentencepiece_length=5 --num_sub_iterations=5 --split_by_number=False --max_sentence_length=30000

sentencepiece_trainer.cc(77) LOG(INFO) Starts training with : 
trainer_spec {
  input: bn-speech.txt
  input_format: text
  model_prefix: spm_800
  model_type: BPE
  vocab_size: 800
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 30000
  num_threads: 16
  num_sub_iterations: 5
  max_sentencepiece_length: 5
  split_by_unicode_script: 1
  split_by_number: 0
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 1
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 1
  hard_vocab_limit: 0
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_privacy_noise_level: 0
  dif

In [16]:
!spm_train --input=bn-speech.txt --input_format=text --model_prefix=spm_1200  --vocab_size=1200 --model_type=bpe \
--character_coverage=1 --random_seed=10  num_threads=4 --pad_id=0 --unk_id=3 --bos_id=1 --eos_id=2 \
--allow_whitespace_only_pieces=True --train_extremely_large_corpus=True --hard_vocab_limit=False \
--max_sentencepiece_length=8 --num_sub_iterations=5 --split_by_number=False --max_sentence_length=30000

sentencepiece_trainer.cc(77) LOG(INFO) Starts training with : 
trainer_spec {
  input: bn-speech.txt
  input_format: text
  model_prefix: spm_1200
  model_type: BPE
  vocab_size: 1200
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 30000
  num_threads: 16
  num_sub_iterations: 5
  max_sentencepiece_length: 8
  split_by_unicode_script: 1
  split_by_number: 0
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 1
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 1
  hard_vocab_limit: 0
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_privacy_noise_level: 0
  d

In [17]:
!spm_train --input=bn-speech.txt --input_format=text --model_prefix=spm_2000  --vocab_size=2000 --model_type=bpe \
--character_coverage=1 --random_seed=10  num_threads=4 --pad_id=0 --unk_id=3 --bos_id=1 --eos_id=2 \
--allow_whitespace_only_pieces=True --train_extremely_large_corpus=True --hard_vocab_limit=False \
--max_sentencepiece_length=8 --num_sub_iterations=5 --split_by_number=False --max_sentence_length=30000

sentencepiece_trainer.cc(77) LOG(INFO) Starts training with : 
trainer_spec {
  input: bn-speech.txt
  input_format: text
  model_prefix: spm_2000
  model_type: BPE
  vocab_size: 2000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 30000
  num_threads: 16
  num_sub_iterations: 5
  max_sentencepiece_length: 8
  split_by_unicode_script: 1
  split_by_number: 0
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 1
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 1
  hard_vocab_limit: 0
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_privacy_noise_level: 0
  d

In [ ]:
tokenizer = SpeechT5Tokenizer('/kaggle/working/spm.model', unk_token="<unk>", pad_token="<pad>", eos_token='</s>', bos_token='<s>', )
print(tokenizer.vocab_size)
print(tokenizer.get_vocab())


In [ ]:
tokenizer.tokenize('<G-S>')

In [ ]:
from transformers import SpeechT5Processor, SpeechT5Tokenizer

In [ ]:
processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_asr")

In [ ]:
processor.tokenizer.vocab_size

In [ ]:
print(processor.tokenizer.tokenize('Hello, how are you'))

In [ ]:
processor.tokenizer.sp_model.load('/kaggle/working/spm.model')

In [ ]:
print(processor.tokenizer.get_vocab())

In [ ]:
processor.tokenizer.vocab_size

In [ ]:
print(processor.tokenizer.tokenize('Hello, how are you?'))